# LTI Systems — A Visual Guide

**Linear Time-Invariant (LTI)** systems underlie signal processing, control engineering, and communications.
Each section pairs core equations with an interactive visualization: move the sliders to build intuition directly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import ipywidgets as widgets
from ipywidgets import interact
%matplotlib inline

plt.rcParams.update({
    'figure.figsize': (11, 4),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
    'font.size': 11,
})
print("Setup done ✓")

Setup done ✓


## 1 · The Two Defining Properties

A system $\mathcal{H}$ is **LTI** when it satisfies both:

**Linearity (Superposition)**
$$\mathcal{H}\{a\,x_1(t) + b\,x_2(t)\} = a\,\mathcal{H}\{x_1(t)\} + b\,\mathcal{H}\{x_2(t)\}$$

**Time Invariance**
$$x(t) \to y(t) \;\Rightarrow\; x(t-\tau) \to y(t-\tau)$$

Running example throughout: the **RC low-pass filter** $\;\dot{y} + y/RC = x/RC$, time constant $\tau = RC$.

Vary $a$ and $b$ below. The error between the direct response and the superposed one stays at machine precision — linearity is exact.

In [ ]:
def rc_response(x, t, tau=1.0):
    sys = signal.TransferFunction([1/tau], [1, 1/tau])
    _, y, _ = signal.lsim(sys, x, t)
    return y

def plot_superposition(a=1.0, b=0.5):
    t  = np.linspace(0, 10, 1000)
    x1 = np.sin(t)
    x2 = np.where((t >= 2) & (t <= 5), 1.0, 0.0)

    y1, y2   = rc_response(x1, t), rc_response(x2, t)
    y_direct = rc_response(a*x1 + b*x2, t)
    y_super  = a*y1 + b*y2
    err      = np.abs(y_direct - y_super)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(t, a*x1 + b*x2, 'C0')
    axes[0].set_title(f'Input:  {a:.1f}·x₁  +  {b:.1f}·x₂')
    axes[0].set_xlabel('t (s)')

    axes[1].plot(t, y_direct, 'C1',    label='Direct response')
    axes[1].plot(t, y_super,  'C2--',  alpha=0.8, label=f'{a:.1f}·y₁ + {b:.1f}·y₂')
    axes[1].set_title('Superposition check')
    axes[1].set_xlabel('t (s)')
    axes[1].legend(fontsize=9)

    axes[2].plot(t, err, 'C3')
    axes[2].set_title(f'|error|  max = {err.max():.1e}  (numerical noise only)')
    axes[2].set_xlabel('t (s)')

    plt.tight_layout()
    plt.show()

interact(plot_superposition,
         a=widgets.FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.1, description='a:'),
         b=widgets.FloatSlider(value=0.5, min=-2.0, max=2.0, step=0.1, description='b:'))

interactive(children=(FloatSlider(value=1.0, description='a:', max=2.0, min=-2.0), FloatSlider(value=0.5, desc…

<function __main__.plot_superposition(a=1.0, b=0.5)>

## 2 · Impulse Response $h(t)$

The **impulse response** is the output when the input is a Dirac delta $\delta(t)$ — a unit-area pulse of zero width.

Because any signal decomposes into scaled, shifted impulses, $h(t)$ alone **fully characterises** the system for any input.

For the RC filter:
$$h(t) = \frac{1}{\tau}\,e^{-t/\tau}\,u(t)$$

- At $t = \tau$: $h$ drops to $e^{-1} \approx 37\%$ of its peak — the canonical definition of the **time constant**
- DC gain: $\displaystyle\int_0^\infty h(t)\,dt = H(0) = 1$
- Smaller $\tau$ → faster decay, wider bandwidth

In [ ]:
def plot_impulse_response(tau=1.0):
    t = np.linspace(0, 6*tau, 1000)
    h = np.exp(-t/tau) / tau

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t, h, 'C0', label=fr'h(t) = (1/τ) exp(−t/τ),   τ = {tau:.2f} s')
    ax.axvline(tau, color='C1', linestyle='--', alpha=0.85,
               label=f't = τ = {tau:.2f} s  →  h drops to 37 %')
    ax.axhline(np.exp(-1)/tau, color='C1', linestyle=':', alpha=0.5)
    area = np.trapezoid(h, t)
    ax.fill_between(t, h, alpha=0.13, color='C0',
                    label=f'Area = {area:.4f}  (DC gain)')
    ax.set_xlabel('t (s)')
    ax.set_ylabel('h(t)')
    ax.set_title('RC Filter — Impulse Response')
    ax.legend()
    plt.tight_layout()
    plt.show()

interact(plot_impulse_response,
         tau=widgets.FloatSlider(value=1.0, min=0.1, max=4.0, step=0.1,
                                 description='τ (s):',
                                 style={'description_width': 'initial'}))

interactive(children=(FloatSlider(value=1.0, description='τ (s):', max=4.0, min=0.1, style=SliderStyle(descrip…

<function __main__.plot_impulse_response(tau=1.0)>

## 3 · Convolution — Assembling the Output

Any input is a continuum of scaled, shifted impulses.
Apply linearity and time invariance to each one:

$$\boxed{y(t) = \int_{-\infty}^{\infty} x(\tau)\,h(t-\tau)\,d\tau} \;=\; (x * h)(t)$$

**"Flip and slide" reading at a fixed $t$:**

1. **Flip** $h(\tau) \to h(-\tau)$
2. **Slide** to position $t$: $h(t-\tau)$
3. **Multiply** by $x(\tau)$ and integrate — that area is $y(t)$

Drag the slider: each position of $t$ picks a different weighted slice of the input history.

In [ ]:
_t      = np.linspace(0, 8, 3000)
_tau_rc = 1.0
_x      = np.where((_t >= 0) & (_t <= 2), 1.0, 0.0)
_h      = np.exp(-_t / _tau_rc)
_y      = np.convolve(_x, _h)[:len(_t)] * (_t[1] - _t[0])

def plot_convolution(t_val=2.0):
    t, tau_rc, x, y = _t, _tau_rc, _x, _y

    h_slide = np.where(t <= t_val, np.exp(-(t_val - t)/tau_rc), 0.0)
    product = x * h_slide
    y_at_t  = np.trapezoid(product, t)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    ax = axes[0]
    ax.plot(t, x, 'C0', label='x(τ) — rect pulse')
    ax.plot(t, h_slide, 'C1', label=f'h({t_val:.1f} − τ)')
    ax.fill_between(t, product, alpha=0.25, color='C2', label='x(τ) · h(t−τ)')
    ax.set_xlim(0, 8); ax.set_ylim(-0.05, 1.4)
    ax.set_xlabel('τ'); ax.set_title(f'Flip & Slide   t = {t_val:.1f} s')
    ax.legend(fontsize=9)

    ax = axes[1]
    ax.fill_between(t, product, alpha=0.4, color='C2')
    ax.plot(t, product, 'C2')
    ax.set_xlim(0, 8); ax.set_ylim(-0.05, 1.4)
    ax.set_xlabel('τ')
    ax.set_title(f'Integrand   →   area = y({t_val:.1f}) = {y_at_t:.3f}')

    ax = axes[2]
    ax.plot(t, y, 'C3', label='y(t) = x ∗ h')
    ax.axvline(t_val, color='k', linestyle='--', alpha=0.5)
    ax.scatter([t_val], [y_at_t], s=90, color='C2', zorder=5,
               label=f'y({t_val:.1f}) = {y_at_t:.2f}')
    ax.set_xlim(0, 8); ax.set_ylim(bottom=-0.05)
    ax.set_xlabel('t'); ax.set_title('Output  y(t)')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

interact(plot_convolution,
         t_val=widgets.FloatSlider(value=2.0, min=0.0, max=8.0, step=0.1,
                                   description='t (s):',
                                   style={'description_width': 'initial'}))

interactive(children=(FloatSlider(value=2.0, description='t (s):', max=8.0, style=SliderStyle(description_widt…

<function __main__.plot_convolution(t_val=2.0)>

## 4 · Transfer Function $H(s)$ — Poles & Zeros

The Laplace transform converts convolution into multiplication:
$$Y(s) = H(s)\cdot X(s) \qquad H(s) = \mathcal{L}\{h(t)\}$$

For a **second-order system** (RLC circuit, mass-spring-damper):
$$H(s) = \frac{\omega_n^2}{s^2 + 2\zeta\,\omega_n\,s + \omega_n^2}$$

Poles: $s_{1,2} = \omega_n\!\left(-\zeta \pm j\sqrt{1-\zeta^2}\right)$

| $\zeta$ | Regime | Pole character |
|---------|--------|----------------|
| $< 1$ | Underdamped — oscillates | Complex conjugate |
| $= 1$ | Critically damped | Real, repeated |
| $> 1$ | Overdamped — no overshoot | Two real |

**Stability** ↔ all poles in the **left half-plane** $\bigl(\operatorname{Re}(s) < 0\bigr)$.

In [ ]:
def plot_second_order(zeta=0.3, omega_n=2.0):
    sys_tf = signal.TransferFunction([omega_n**2], [1, 2*zeta*omega_n, omega_n**2])
    poles  = np.roots([1, 2*zeta*omega_n, omega_n**2])

    settle = 5.0 / max(zeta*omega_n, 0.05)
    t_step = np.linspace(0, min(settle, 60), 2000)
    _, y_step = signal.step(sys_tf, T=t_step)

    if zeta < 0.999:
        regime = 'Underdamped'
    elif zeta < 1.001:
        regime = 'Critically damped'
    else:
        regime = 'Overdamped'

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    lim = omega_n * 1.6
    ax = axes[0]
    ax.axhline(0, color='k', lw=0.8); ax.axvline(0, color='k', lw=0.8)
    ax.axvspan(-lim*2, 0, alpha=0.06, color='C2', label='Stable region  Re(s) < 0')
    ax.scatter(poles.real, poles.imag,
               marker='x', s=180, color='C3', linewidths=2.5, zorder=5, label='Poles')
    ax.set_xlim(-lim*1.8, lim*0.5); ax.set_ylim(-lim*1.2, lim*1.2)
    ax.set_xlabel('Re(s)'); ax.set_ylabel('Im(s)')
    ax.set_title(f'Pole map   ζ = {zeta:.2f},  ωₙ = {omega_n:.1f} rad/s')
    ax.legend(fontsize=9)

    ax = axes[1]
    ax.plot(t_step, y_step, 'C0')
    ax.axhline(1.0, color='k', linestyle='--', alpha=0.4, label='y(∞) = 1')
    ax.set_xlabel('t (s)'); ax.set_ylabel('y(t)')
    ax.set_title(f'Step response — {regime}')
    ax.legend()

    plt.tight_layout()
    plt.show()

interact(plot_second_order,
         zeta=widgets.FloatSlider(value=0.3, min=0.05, max=2.5, step=0.05,
                                  description='ζ:',
                                  style={'description_width': 'initial'}),
         omega_n=widgets.FloatSlider(value=2.0, min=0.5, max=6.0, step=0.1,
                                     description='ωₙ (rad/s):',
                                     style={'description_width': 'initial'}))

interactive(children=(FloatSlider(value=0.3, description='ζ:', max=2.5, min=0.05, step=0.05, style=SliderStyle…

<function __main__.plot_second_order(zeta=0.3, omega_n=2.0)>

## 5 · Frequency Response and Bode Plot

Evaluate $H(s)$ on the imaginary axis $s = j\omega$:
$$H(j\omega) = |H(j\omega)|\,e^{\,j\angle H(j\omega)}$$

A sinusoidal input $x(t) = A\sin(\omega t)$ produces in **steady state**:
$$y_{ss}(t) = A\,|H(j\omega)|\;\sin\!\bigl(\omega t + \angle H(j\omega)\bigr)$$

The **Bode plot** displays both quantities on a log frequency axis:
- **Magnitude** $20\log_{10}|H(j\omega)|$ in dB — amplification or attenuation per frequency
- **Phase** $\angle H(j\omega)$ in degrees — time delay introduced at each frequency
- **Bandwidth** $\omega_{-3\text{dB}}$: power halved ($|H| = 1/\sqrt{2}$, i.e. $-3$ dB)

Low $\zeta$ near $\omega_n$ produces a **resonance peak** in the magnitude.

In [ ]:
def plot_bode(zeta=0.3, omega_n=2.0):
    sys_tf = signal.TransferFunction([omega_n**2], [1, 2*zeta*omega_n, omega_n**2])
    w      = np.logspace(-1, 2, 3000)
    w_b, mag_dB, phase_deg = signal.bode(sys_tf, w=w)

    peak_dB = mag_dB.max()
    bw_idx  = np.where(mag_dB <= -3)[0]
    bw_str  = (f'ω₋₃dB = {w_b[bw_idx[0]]:.2f} rad/s' if len(bw_idx) else 'BW > 100')

    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

    ax = axes[0]
    ax.semilogx(w_b, mag_dB, 'C0')
    ax.axhline(-3, color='C1', linestyle='--', alpha=0.7, label='−3 dB')
    ax.axvline(omega_n, color='C2', linestyle='--', alpha=0.65,
               label=f'ωₙ = {omega_n:.1f} rad/s')
    if len(bw_idx):
        ax.axvline(w_b[bw_idx[0]], color='C1', linestyle=':', alpha=0.7, label=bw_str)
    ax.set_ylabel('Magnitude (dB)')
    ax.set_title(f'Bode Plot   ζ = {zeta:.2f},  ωₙ = {omega_n:.1f} rad/s   (peak = {peak_dB:.1f} dB)')
    ax.legend(fontsize=9)

    ax = axes[1]
    ax.semilogx(w_b, phase_deg, 'C3')
    ax.axhline(-90, color='gray', linestyle=':', alpha=0.5, label='−90°')
    ax.axvline(omega_n, color='C2', linestyle='--', alpha=0.65,
               label=f'ωₙ = {omega_n:.1f}')
    ax.set_xlabel('Frequency (rad/s)')
    ax.set_ylabel('Phase (°)')
    ax.set_yticks([-180, -135, -90, -45, 0])
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

interact(plot_bode,
         zeta=widgets.FloatSlider(value=0.3, min=0.05, max=2.5, step=0.05,
                                  description='ζ:',
                                  style={'description_width': 'initial'}),
         omega_n=widgets.FloatSlider(value=2.0, min=0.5, max=6.0, step=0.1,
                                     description='ωₙ (rad/s):',
                                     style={'description_width': 'initial'}))

interactive(children=(FloatSlider(value=0.3, description='ζ:', max=2.5, min=0.05, step=0.05, style=SliderStyle…

<function __main__.plot_bode(zeta=0.3, omega_n=2.0)>

## 6 · Sinusoidal Steady-State Verification

The Bode plot makes a testable prediction: after transients decay, a sinusoidal input at $\omega$ produces output at the **same frequency**, scaled and shifted exactly by $H(j\omega)$.

The upper panel overlays the simulated response (`lsim`) with the analytical steady-state $y_{ss}(t) = |H(j\omega)|\sin(\omega t + \angle H)$.
The residual in the lower panel decays to zero once the transient fades — confirming the frequency-domain picture.

Try setting $\omega \approx \omega_n$ with low $\zeta$ to see resonance amplification, then push $\omega \gg \omega_n$ to observe attenuation and phase lag.

In [7]:
def plot_sinusoidal_ss(omega=1.0, zeta=0.3, omega_n=2.0):
    sys_tf = signal.TransferFunction([omega_n**2], [1, 2*zeta*omega_n, omega_n**2])

    _, H_val  = signal.freqs(sys_tf.num, sys_tf.den, worN=[omega])
    gain      = float(np.abs(H_val[0]))
    phase_rad = float(np.angle(H_val[0]))

    settle = 6.0 / max(zeta*omega_n, 0.1)
    t_end  = min(settle + 5*2*np.pi/omega, 120)
    t      = np.linspace(0, t_end, 5000)
    x      = np.sin(omega * t)
    _, y_sim, _ = signal.lsim(sys_tf, x, t)
    y_ss   = gain * np.sin(omega*t + phase_rad)

    cutoff = int(0.55 * len(t))
    residual = y_sim - y_ss

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    ax = axes[0]
    ax.plot(t, x,     'C0', alpha=0.6, label='x(t) = sin(ωt)')
    ax.plot(t, y_sim, 'C1',            label='y(t) simulated')
    ax.plot(t, y_ss,  'C2--', alpha=0.9,
            label=f'y_ss predicted   gain = {gain:.2f},  Δφ = {np.degrees(phase_rad):.1f}°')
    ax.axvline(t[cutoff], color='gray', linestyle=':', alpha=0.55,
               label='transient / SS boundary')
    ax.set_ylabel('Amplitude')
    ax.set_title(f'ω = {omega:.2f} rad/s,   ζ = {zeta:.2f},   ωₙ = {omega_n:.1f} rad/s')
    ax.legend(fontsize=9)

    ax = axes[1]
    ax.plot(t, residual, 'C4')
    ax.axvline(t[cutoff], color='gray', linestyle=':', alpha=0.55)
    ss_max = np.abs(residual[cutoff:]).max()
    ax.set_xlabel('t (s)')
    ax.set_ylabel('y_sim − y_ss')
    ax.set_title(f'Residual   (SS amplitude = {ss_max:.2e}  → 0 in steady state)')

    plt.tight_layout()
    plt.show()

interact(plot_sinusoidal_ss,
         omega=widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1,
                                   description='ω (rad/s):',
                                   style={'description_width': 'initial'}),
         zeta=widgets.FloatSlider(value=0.3, min=0.05, max=2.0, step=0.05,
                                  description='ζ:',
                                  style={'description_width': 'initial'}),
         omega_n=widgets.FloatSlider(value=2.0, min=0.5, max=6.0, step=0.1,
                                     description='ωₙ (rad/s):',
                                     style={'description_width': 'initial'}))

interactive(children=(FloatSlider(value=1.0, description='ω (rad/s):', max=10.0, min=0.1, style=SliderStyle(de…

<function __main__.plot_sinusoidal_ss(omega=1.0, zeta=0.3, omega_n=2.0)>